In [1]:
# rnn_train_valuation_seq.py
# Trains a simple GRU/LSTM on valuation-sequence NPZ:
#   X_seq:    (N, T, F_seq)
#   X_static: (N, F_static)
#   y:        (N,)  (log target)
#   target_date: (N,) datetime64[ns]

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# -----------------------
# 1) Load dataset
# -----------------------
npz_path = "../Data_Processed/valuation_seq_rnn_dataset.npz"

d = np.load(npz_path, allow_pickle=True)
X_seq = d["X_seq"].astype(np.float32)
X_static = d["X_static"].astype(np.float32)
y = d["y"].astype(np.float32)
target_date = pd.to_datetime(d["target_date"])

print("Loaded:")
print("  X_seq:", X_seq.shape)
print("  X_static:", X_static.shape)
print("  y:", y.shape)
print("  target_date:", target_date.min(), "->", target_date.max())

# -----------------------
# 2) Time-based split (no leakage)
# -----------------------
train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2023-12-31")

train_idx = np.where(target_date <= train_end)[0]
val_idx = np.where((target_date > train_end) & (target_date <= val_end))[0]
test_idx = np.where(target_date > val_end)[0]

print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))

# -----------------------
# 3) Standardize features using TRAIN ONLY
#    (very important for neural nets)
# -----------------------
def fit_standardizer_seq(X_seq_train):
    # X_seq_train: (N, T, F)
    # compute mean/std over N and T for each feature
    mu = X_seq_train.reshape(-1, X_seq_train.shape[-1]).mean(axis=0)
    sigma = X_seq_train.reshape(-1, X_seq_train.shape[-1]).std(axis=0)
    sigma = np.where(sigma < 1e-8, 1.0, sigma)
    return mu.astype(np.float32), sigma.astype(np.float32)

def apply_standardizer_seq(X_seq, mu, sigma):
    return (X_seq - mu[None, None, :]) / sigma[None, None, :]

def fit_standardizer_tab(X_tab_train):
    mu = X_tab_train.mean(axis=0)
    sigma = X_tab_train.std(axis=0)
    sigma = np.where(sigma < 1e-8, 1.0, sigma)
    return mu.astype(np.float32), sigma.astype(np.float32)

def apply_standardizer_tab(X_tab, mu, sigma):
    return (X_tab - mu[None, :]) / sigma[None, :]

seq_mu, seq_sigma = fit_standardizer_seq(X_seq[train_idx])
static_mu, static_sigma = fit_standardizer_tab(X_static[train_idx])

X_seq = apply_standardizer_seq(X_seq, seq_mu, seq_sigma)
X_static = apply_standardizer_tab(X_static, static_mu, static_sigma)

print("Standardization done (train-only).")

# -----------------------
# 4) Torch Dataset
# -----------------------
class ValSeqDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_static[idx], self.y[idx]

train_ds = ValSeqDataset(X_seq[train_idx], X_static[train_idx], y[train_idx])
val_ds = ValSeqDataset(X_seq[val_idx], X_static[val_idx], y[val_idx])
test_ds = ValSeqDataset(X_seq[test_idx], X_static[test_idx], y[test_idx])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

# -----------------------
# 5) Model (choose GRU or LSTM)
# -----------------------
class GRURegressor(nn.Module):
    def __init__(self, seq_dim, static_dim, hidden_dim=64):
        super().__init__()
        self.rnn = nn.GRU(seq_dim, hidden_dim, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + static_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_static):
        _, h = self.rnn(x_seq)      # h: (1, B, H)
        h = h.squeeze(0)            # (B, H)
        x = torch.cat([h, x_static], dim=1)
        return self.head(x)

class LSTMRegressor(nn.Module):
    def __init__(self, seq_dim, static_dim, hidden_dim=64):
        super().__init__()
        self.rnn = nn.LSTM(seq_dim, hidden_dim, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + static_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_static):
        _, (h, c) = self.rnn(x_seq)  # h: (1, B, H)
        h = h.squeeze(0)
        x = torch.cat([h, x_static], dim=1)
        return self.head(x)

# Pick one:
USE_LSTM = False

seq_dim = X_seq.shape[2]
static_dim = X_static.shape[1]

device = "cuda" if torch.cuda.is_available() else "cpu"
model = (LSTMRegressor(seq_dim, static_dim, 64) if USE_LSTM else GRURegressor(seq_dim, static_dim, 64)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# -----------------------
# 6) Train / eval loops
# -----------------------
def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)

    total = 0.0
    n = 0

    for x_seq_b, x_static_b, y_b in loader:
        x_seq_b = x_seq_b.to(device)
        x_static_b = x_static_b.to(device)
        y_b = y_b.to(device)

        pred = model(x_seq_b, x_static_b)
        loss = loss_fn(pred, y_b)

        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        bs = y_b.size(0)
        total += loss.item() * bs
        n += bs

    return total / max(n, 1)

best_val = float("inf")
best_state = None

EPOCHS = 15
for epoch in range(1, EPOCHS + 1):
    train_mse = run_epoch(model, train_loader, optimizer)
    val_mse = run_epoch(model, val_loader, optimizer=None)

    print(f"Epoch {epoch:02d} | train MSE: {train_mse:.4f} | val MSE: {val_mse:.4f}")

    if val_mse < best_val:
        best_val = val_mse
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print("Best val MSE:", best_val)

if best_state is not None:
    model.load_state_dict(best_state)

test_mse = run_epoch(model, test_loader, optimizer=None)
print("Test MSE (log scale):", test_mse)

# Optional: convert to "multiplicative error factor" approx
rmse_log = float(np.sqrt(test_mse))
print("Test RMSE (log units):", rmse_log)
print("Approx multiplicative factor exp(RMSE):", float(np.exp(rmse_log)))


Loaded:
  X_seq: (337430, 5, 11)
  X_static: (337430, 10)
  y: (337430,)
  target_date: 2005-04-03 00:00:00 -> 2025-03-31 00:00:00
Split sizes: 251794 67018 18618
Standardization done (train-only).
Epoch 01 | train MSE: 8.0671 | val MSE: 0.0821
Epoch 02 | train MSE: 0.0888 | val MSE: 0.0808
Epoch 03 | train MSE: 0.0890 | val MSE: 0.0824
Epoch 04 | train MSE: 0.0880 | val MSE: 0.0784


KeyboardInterrupt: 